# Lakehouse Architecture

The **data lakehouse** is the architectural pattern at the heart of the Databricks platform. It combines the low-cost, flexible storage of a data lake with the reliability and performance guarantees of a data warehouse — on a single system.

In this notebook we'll cover:
- Why lakehouses exist — the limitations of warehouses and lakes
- The key properties that define a lakehouse
- How the Databricks Lakehouse Platform is structured
- The role Delta Lake plays as the storage foundation

## The Evolution: Warehouse → Lake → Lakehouse

| | Data Warehouse | Data Lake | Data Lakehouse |
|---|---|---|---|
| **Storage** | Proprietary, expensive | Object store (S3/ADLS/GCS), cheap | Object store, cheap |
| **Data types** | Structured only | Structured, semi-structured, unstructured | All types |
| **Schema** | Schema-on-write (enforced upfront) | Schema-on-read (no enforcement) | Schema-on-write with flexibility |
| **ACID transactions** | Yes | No | Yes (via Delta Lake) |
| **BI / SQL** | Excellent | Poor | Excellent |
| **ML / Data Science** | Poor | Good | Excellent |
| **Openness** | Proprietary formats | Open formats (Parquet, ORC) | Open formats |
| **Cost** | High | Low | Low |

## Why Data Warehouses Fall Short

Data warehouses were built for structured, relational data and SQL analytics. They work well for that — but they have hard limits:

- **Cost** — proprietary storage is expensive at scale
- **No unstructured data** — images, video, text, JSON blobs don't fit
- **ML is second-class** — you must export data out to a separate ML platform
- **Closed formats** — data is locked in vendor-specific storage, making portability painful
- **Streaming is bolt-on** — near-real-time ingestion requires extra infrastructure

The typical enterprise response was to bolt a data lake alongside the warehouse — creating **two separate systems** that teams had to keep in sync. This two-tier architecture added complexity, data duplication, and staleness.

## Why Data Lakes Fall Short

Data lakes solved the cost and flexibility problems — cheap object storage, any file format, great for ML. But they introduced a different set of problems:

- **No ACID transactions** — concurrent writes could corrupt data; partial job failures left inconsistent state
- **No schema enforcement** — bad data would land silently; downstream queries would fail or return wrong results
- **Poor BI performance** — no indexes, no statistics, no caching; SQL tools like Tableau would time out on raw Parquet
- **No audit trail** — you couldn't see who changed what, or roll back to a previous state
- **Data swamps** — without governance, lakes accumulate stale, undocumented, untrusted data

The lake gave you flexibility. The warehouse gave you reliability. The lakehouse gives you both.

## Lakehouse Key Properties

A true lakehouse has six defining properties:

### 1. ACID Transactions
Concurrent readers and writers don't corrupt each other's data. A failed job doesn't leave partial results. This is implemented through a **transaction log** (Delta Lake's `_delta_log/`).

### 2. Schema Enforcement & Evolution
Data written to a table must match its schema — bad records are rejected, not silently accepted. Schema can evolve (add new columns) without breaking existing queries.

### 3. BI & SQL Support
Standard SQL tools (Databricks SQL, Tableau, Power BI) can query lakehouse tables directly, with performance comparable to a warehouse — because lakehouse tables have statistics, indexing (Z-ordering), and caching.

### 4. Decoupled Storage & Compute
Data lives in open-format files on cheap cloud object storage (S3, ADLS, GCS). Compute clusters can scale independently — or be shut down entirely when not needed — without touching the data.

### 5. Openness
Storage is in open formats (Delta Lake, Apache Parquet under the hood). Other engines (Spark, Presto, Trino, pandas) can read the same data without going through a proprietary API.

### 6. End-to-End Workloads
The same platform handles ingestion, ELT, SQL analytics, and ML model training. No data copying between systems.

## The Databricks Lakehouse Platform

Databricks is the commercial platform built around the lakehouse pattern. It layers several components on top of cloud object storage:

```
┌─────────────────────────────────────────────────────────┐
│                  Databricks Platform                    │
│                                                         │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌────────┐  │
│  │Databricks│  │Databricks│  │  MLflow  │  │ Unity  │  │
│  │  Notebooks│  │   SQL    │  │(ML/Expts)│  │Catalog │  │
│  └──────────┘  └──────────┘  └──────────┘  └────────┘  │
│                                                         │
│              Apache Spark Runtime                       │
│                                                         │
│                   Delta Lake                            │
│        (transaction log + Parquet files)                │
│                                                         │
│         Cloud Object Storage (S3 / ADLS / GCS)         │
└─────────────────────────────────────────────────────────┘
```

### Delta Lake
The storage layer. Adds ACID transactions, schema enforcement, time travel, and performance optimizations (OPTIMIZE, Z-ORDER, VACUUM) on top of Parquet files. Covered in depth in `03-delta-lake-on-databricks.ipynb`.

### Apache Spark Runtime
The compute engine. Databricks ships an optimized version of Spark (Databricks Runtime / DBR) with additional performance improvements like Photon — a native vectorized query engine written in C++.

### Databricks Notebooks
Interactive, collaborative notebooks that support Python, Scala, SQL, and R — all in one notebook. Multiple users can co-author in real time.

### Databricks SQL
A SQL warehouse (dedicated compute) for BI and ad-hoc queries. Connects directly to tools like Tableau, Power BI, and Looker via JDBC/ODBC.

### MLflow
Open-source ML lifecycle management. Tracks experiments, packages models, and manages deployments. Built into Databricks.

### Unity Catalog
Centralized governance layer. Manages access control, data lineage, and auditing across all workspaces and all data assets. Covered in `15-unity-catalog.ipynb`.

## Medallion Architecture

The standard data organisation pattern in a Databricks lakehouse is the **medallion architecture** — three layers of progressively cleaner data:

```
Raw source data
      │
      ▼
┌──────────┐     ┌──────────┐     ┌──────────┐
│  Bronze  │────▶│  Silver  │────▶│   Gold   │
│  (raw)   │     │ (cleaned)│     │(aggregated│
└──────────┘     └──────────┘     └──────────┘
```

| Layer | Purpose | Example |
|-------|---------|--------|
| **Bronze** | Raw ingestion — data as-is from the source | JSON events from Kafka, CSV uploads, CDC feeds |
| **Silver** | Cleaned, validated, joined — conformed to a schema | Deduplicated orders with valid customer IDs |
| **Gold** | Business-level aggregations — ready for BI or ML | Daily revenue by region, customer churn features |

Each layer is a set of Delta tables. Bronze is append-only (never delete raw data). Silver applies business rules. Gold is what analysts and models consume.

This pattern is also called **multi-hop architecture** — data hops from layer to layer, getting refined at each step.

## Hands-On: Creating Lakehouse Tables

In Databricks, `spark` is pre-configured — no `SparkSession.builder` needed. The following examples use the SQL API and PySpark to create and query Delta tables, demonstrating the medallion pattern.

In [ ]:
# Create a schema (database) to hold our medallion tables
spark.sql("CREATE SCHEMA IF NOT EXISTS lakehouse_demo")
spark.sql("USE lakehouse_demo")

In [ ]:
# Bronze layer — raw orders as they arrived (no cleaning)
spark.sql("""
  CREATE OR REPLACE TABLE bronze_orders (
    order_id    STRING,
    customer_id STRING,
    amount      STRING,   -- raw string, may contain garbage
    status      STRING,
    event_ts    STRING
  ) USING DELTA
""")

spark.sql("""
  INSERT INTO bronze_orders VALUES
    ('ORD001', 'CUST01', '120.50',  'completed', '2024-01-15 09:00:00'),
    ('ORD002', 'CUST02', 'N/A',     'pending',   '2024-01-15 09:05:00'),
    ('ORD003', 'CUST01', '340.00',  'completed', '2024-01-15 09:10:00'),
    ('ORD001', 'CUST01', '120.50',  'completed', '2024-01-15 09:00:00')  -- duplicate
""")

spark.sql("SELECT * FROM bronze_orders").show()

In [ ]:
# Silver layer — cleaned: cast types, remove duplicates, drop invalid amounts
spark.sql("""
  CREATE OR REPLACE TABLE silver_orders
  USING DELTA AS
  SELECT DISTINCT
    order_id,
    customer_id,
    CAST(amount AS DOUBLE) AS amount,
    status,
    CAST(event_ts AS TIMESTAMP) AS event_ts
  FROM bronze_orders
  WHERE amount != 'N/A'
""")

spark.sql("SELECT * FROM silver_orders").show()

In [ ]:
# Gold layer — aggregated: revenue per customer, ready for BI
spark.sql("""
  CREATE OR REPLACE TABLE gold_customer_revenue
  USING DELTA AS
  SELECT
    customer_id,
    COUNT(*)        AS total_orders,
    SUM(amount)     AS total_revenue,
    AVG(amount)     AS avg_order_value
  FROM silver_orders
  WHERE status = 'completed'
  GROUP BY customer_id
""")

spark.sql("SELECT * FROM gold_customer_revenue").show()

In [ ]:
# All three tables are Delta — we get time travel for free
# Check the transaction history of the silver table
spark.sql("DESCRIBE HISTORY silver_orders").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

In [ ]:
# Cleanup
spark.sql("DROP SCHEMA IF EXISTS lakehouse_demo CASCADE")

## Summary

- A **data lakehouse** combines cheap object-store flexibility with warehouse-grade reliability — ACID transactions, schema enforcement, and BI performance on open-format data.
- The lakehouse emerged to fix the two-tier warehouse + lake architecture that forced teams to duplicate and synchronise data across systems.
- The **Databricks Lakehouse Platform** layers Notebooks, Databricks SQL, MLflow, and Unity Catalog over Apache Spark, with **Delta Lake** as the transactional storage foundation.
- The **medallion architecture** (Bronze → Silver → Gold) is the standard pattern for organising data: raw ingestion, cleaned/conformed, aggregated for consumption.
- All layers are Delta tables — so you get time travel, schema enforcement, and ACID guarantees at every hop.

Next: `02-workspace-clusters.ipynb` — how Databricks workspaces are structured and how clusters are configured for different workloads.